In [1]:
#load imports

%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../..')


import numpy as np
import src.demo as demo
import src.viz_utils as viz_utils
import src.utils as utils
import importlib
from src.utils import CanonPart, CanonPartMetadata, get_pointcloud_in_cam_frame, transform_cloud_to_base, remove_outliers
from PIL import Image
import torch
import open3d as o3d
import open3d.visualization.gui as gui
import copy as cp
import pickle
import matplotlib
from src.ndf_interface import NDFInterface

pybullet build time: May 20 2022 19:45:31


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
#load pointclouds, interaction points, meshes, and program

#Load the files and camera information
root_file = '/home/rthomp12/fewshot/scripts/experiment_notebooks/whole_bowl_on_mug_test_20250113-045403_pink_bowl_eggnog_mug'
demo_folder = '/home/rthomp12/fewshot/scripts/experiment_notebooks/bowl_on_mug_20250113-033003'

save_name = f'{root_file}/init_scene_pcls.npz'
scene_pcls = np.load(save_name)
scene_pcls = {k: scene_pcls[k] for k in scene_pcls.keys()}

#Load interaction points and 
demo_file = f'{root_file}/whole_interaction_points.pkl'

demo_file = f'{demo_folder}/whole_interaction_points.pkl'
preplace_demo_file = f'{demo_folder}/whole_preplace_interaction_points.pkl'

transform_name = f'{demo_folder}/ee_transform.npz'
demo_transform = np.load(transform_name)

init_transform = utils.pos_quat_to_transform(demo_transform['init_pos'],
                                             demo_transform['init_quat'])

final_transform = utils.pos_quat_to_transform(demo_transform['final_pos'],
                                              demo_transform['final_quat'])

preplace_transform = utils.pos_quat_to_transform(demo_transform['preplace_pos'],
                                                 demo_transform['preplace_quat'])

inferred_preplace = np.matmul(preplace_transform, np.linalg.inv(final_transform))


warps = np.load(f"{root_file}/whole_initial_scene_warps.npz", allow_pickle=True)
warps = {k: warps[k] for k in warps.keys()}
child_params = warps['child_params'].item()
parent_params = warps['parent_params'].item() #np.load(f"/home/rthomp12/fewshot/scripts/experiment_notebooks/whole_mug_on_tree_test_20250112-203322_peep_mug_flared_rack/whole_initial_scene_warps.npz", allow_pickle=True)['parent_params'].item() #

child_reconstruction = warps['child_reconstructions']

FileNotFoundError: [Errno 2] No such file or directory: '/home/rthomp12/fewshot/scripts/experiment_notebooks/whole_bowl_on_mug_test_20250113-045403_pink_bowl_eggnog_mug/whole_init_scene_pcls.npz'

In [3]:
# Set the kind of demo

# Mug v Rack
# parent_object = 'tree'
# child_object = 'mug'
# parent_model_file = {'tree': '/home/rthomp12/fewshot/part_based_warp_models/whole_syn_rack_easy_20240412-042732',}['tree']
# child_model_file = {'mug': '/home/rthomp12/fewshot/whole_mug_20240502-182007_6',}['mug']

# if 'tree' not in scene_pcls.keys():
#     scene_pcls = {k: utils.farthest_point_sample(scene_pcls[k], min(len(scene_pcls[k]), 500))[0] for k in scene_pcls.keys()}
#     scene_pcls['tree'] = np.concatenate([scene_pcls['trunk'], scene_pcls['branch']])
#     scene_pcls['mug'] = np.concatenate([scene_pcls['cup'], scene_pcls['handle']])
# else:
#     scene_pcls = {k: utils.farthest_point_sample(scene_pcls[k], min(len(scene_pcls[k]), 1000))[0] for k in scene_pcls.keys()}



whole = True
parent_object = 'mug'
child_object = 'whole_teapot'

parent_model_file = {'mug': '/home/rthomp12/fewshot/whole_mug_20240502-182007_6',}['mug']
child_model_file = {'whole_teapot': '/home/rthomp12/fewshot/part_based_warp_models/whole_teapot_20241031-012841_5'}['whole_teapot']

parent_object = "mug"



# # Bowl v Mug

# parent_object = 'mug'
# child_object= 'whole_bowl'


# parent_model_file = {'mug': '/home/rthomp12/fewshot/whole_mug_20240502-182007_6',}['mug']

# child_model_file = {'whole_bowl': '/home/rthomp12/fewshot/part_based_warp_models/whole_bowl_20240426-000022_10'}['whole_bowl']

# parent_object = "mug"




parent_model = CanonPart.from_pickle(parent_model_file)
child_model = CanonPart.from_pickle(child_model_file)

all_names = [parent_object, child_object]

interface = NDFInterface(
        canon_source_path = {child_object: child_model_file},
        canon_target_path = {parent_object: parent_model_file},
        source_whole_name = [child_object],
        target_whole_name = [parent_object]
)

In [11]:
# from distutils.spawn import find_executable

# find_executable('testVHACD')

In [ ]:
trans_predicted = interface.infer_relpose(
        scene_pcls[child_object], 
        scene_pcls[parent_object], 
        None, 
        se3=True, 
        child_params=child_params, 
        parent_params=parent_params, 
        knn_pkl=demo_file,
    )
print("FINAL PLACEMENT~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
print()
print(repr(trans_predicted))
print()

# print("PREPLACE~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

# preplace_trans_predicted = interface.infer_relpose(
#         scene_pcls[child_object], 
#         scene_pcls[parent_object], 
#         None, 
#         se3=True, 
#         child_params=child_params, 
#         parent_params=parent_params, 
#         knn_pkl=preplace_demo_file,
#     )
# print()
# print(repr(preplace_trans_predicted))
# print()

INFO - 2025-01-30 13:26:34,063 - generic - executing: /usr/bin/testVHACD /tmp/0_vnbufsh4.obj -resolution 1000000 -depth 20 -concavity 0.0025 -planeDownsampling 4 -convexhullDownsampling 4 -alpha 0.05 -beta 0.05 -gamma 0.00125 -pca 0 -mode 0 -maxNumVerticesPerCH 256 -minVolumePerCH 0.0001 -convexhullApproximation 1 -oclDeviceID 0
INFO - 2025-01-30 13:26:35,170 - generic - 
[COMPUTE_BOUNDS_OF_INPUT_MESH            ] : 0% : 0% : ComputingBoun[COMPUTE_BOUNDS_OF_INPUT_MESH            ] : 0% : 100% : ComputingBounds
[CREATE_RAYCAST_MESH                     ] : 20% : 0% : Building RaycastMe[CREATE_RAYCAST_MESH                     ] : 20% : 100% : RaycastMesh completed
[VOXELIZING_INPUT_MESH                   ] : 30% : 0% : Voxelizing Input Mes[VOXELIZING_INPUT_MESH                   ] : 30% : 100% : Voxelization complete
[BUILD_INITIAL_CONVEX_HULL               ] : 40% : 0% : Build initial ConvexHu[BUILD_INITIAL_CONVEX_HULL               ] : 40% : 100% : Initial ConvexHull complete
[PERFORMIN

tmp_source.obj


INFO - 2025-01-30 13:26:35,926 - generic - 
[COMPUTE_BOUNDS_OF_INPUT_MESH            ] : 0% : 0% : ComputingBoun[COMPUTE_BOUNDS_OF_INPUT_MESH            ] : 0% : 100% : ComputingBounds
[CREATE_RAYCAST_MESH                     ] : 20% : 0% : Building RaycastMe[CREATE_RAYCAST_MESH                     ] : 20% : 100% : RaycastMesh completed
[VOXELIZING_INPUT_MESH                   ] : 30% : 0% : Voxelizing Input Mes[VOXELIZING_INPUT_MESH                   ] : 30% : 100% : Voxelization complete
[BUILD_INITIAL_CONVEX_HULL               ] : 40% : 0% : Build initial ConvexHu[BUILD_INITIAL_CONVEX_HULL               ] : 40% : 100% : Initial ConvexHull complete
[PERFORMING_DECOMPOSITION                ] : 50% : 0% : Performing recursive decomposition of convex hul[PERFORMING_DECOMPOSITION                ] : 50% : 1% : Performing recursive decomposition of convex hul[PERFORMING_DECOMPOSITION                ] : 50% : 21% : Performing recursive decomposition of convex hulls
[INITIALIZING_CONVEX_HULL

tmp_target.obj
FINAL PLACEMENT~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

array([[ 0.90112997,  0.41691437, -0.11894198,  0.02684566],
       [-0.34715785,  0.85821847,  0.37807736, -0.02566438],
       [ 0.25970408, -0.29940519,  0.91810148, -0.26995903],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])



: 

In [13]:
#visualize with pcd and mesh
transformed_scene_pcls = {}

transformed_scene_pcls[child_object] = utils.transform_pcd(scene_pcls[child_object],
                                             trans_predicted,
                                           )

transformed_scene_pcls[parent_object] = scene_pcls[parent_object]

# transformed_scene_pcls[f'preplace_{child_object}'] = utils.transform_pcd(scene_pcls[child_object],
#                                              preplace_trans_predicted,
#                                            )

#     reconstructions[f'reconstructed_{child_part}'] = \
#         child_part_models[child_part].to_transformed_pcd(child_params[child_part]) 

camera = dict(
    eye=dict(x=1.2, y=-1.2, z=.2),
    center=dict(x=.8,y=.5,z=0)
)

fig = viz_utils.show_pcds_plotly(transformed_scene_pcls, camera=camera)
fig.show()

pickle.dump(fig, open(f'{root_file}/final_place', 'wb'))

In [ ]:


pp = pickle.load(open(preplace_demo_file, 'rb'))
preplace_knns = pp['knns']
preplace_target_indices = pp['target_indices']
preplace_deltas = pp['deltas']

targets_child = None
targets_parent = None

anchors = child_model.to_pcd(child_params)[
    preplace_knns
]
targets_child = np.mean(
    anchors + preplace_deltas, axis=1
)
targets_parent = parent_model.to_pcd(parent_params)[
    preplace_target_indices
] 

child_part_targets = {}   
child_targets_viz = {}
parent_targets_viz = {}


child_transform  = utils.pos_quat_to_transform(child_params.position, 
                                              child_params.quat)
parent_transform  = utils.pos_quat_to_transform(parent_params.position, 
                                          parent_params.quat)
child_targets_viz =  child_targets_viz | {
                     f'child_targets': \
                     utils.transform_pcd(targets_child,
                                         child_transform),
                     }
parent_targets_viz = parent_targets_viz | {
                     f'parent_targets': \
                     utils.transform_pcd(targets_parent,
                                         parent_transform),
                     }

viz_pcls = {'parent':scene_pcls[parent_object]}| child_targets_viz | parent_targets_viz | transformed_scene_pcls

viz_utils.show_pcds_plotly(viz_pcls)

: 